# RUN-001 · Intraday Momentum for ETF — A股复现（510300.SH）

论文：*Beat the Market: An Effective Intraday Momentum Strategy for S&P500 ETF (SPY)*
（Zarattini, Aziz, Barbon；SSRN 4824172，2025-09-22 版）

- 标的：510300.SH 沪深300ETF（华泰柏瑞）；基准：510300 自身 B&H；对照：000300.SH
- 样本：2017-01-03 ~ 2025-12-31（2,186 交易日，分钟数据上限）；日频预热自 2016-11-01
- 决策网格：g30（论文主线 7 时点）/ g15（14 时点）/ g05（42 时点）；变体 A/B/C × 三网格
- 成本：佣金 万1/边 + 滑点 ¥0.001/份/边（1 tick），ETF 免印花税；整手 100 份
- 可执行性：假设已有底仓可 T+0（用户 2026-10-04 裁决）；缓冲需求见 BLOCK-18
- 数据：局域网 MySQL 冻结快照（DT-001，QA 21/21 PASS）；bar 为 END 时间标签
  （09:30 竞价 bar + 09:31..11:30 / 13:01..15:00；信号=标签 τ bar 收盘，成交=下一分钟 bar 开盘）

本 notebook 按 plan_code.md BLOCK-00~18 组织，production 模式干净内核顺序执行。

## BLOCK-00 配置与公共函数

集中参数、路径、常量与公共函数。类型 NO_ARTIFACT。

In [1]:
# BLOCK-00 配置与公共函数
import json
import math
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

ROOT = Path.cwd()
if not (ROOT / "data" / "processed").exists():
    ROOT = Path(r"D:\quantitative_project\复现工程\Intraday_Momentum_for_ETF")
PROC = ROOT / "data" / "processed"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
EXT_DIR = OUT / "ext"

RUN_ID = "RUN-001"
TS_CODE = "510300.SH"
TS_NAME = "沪深300ETF(华泰柏瑞)"
INIT_CAPITAL = 1_000_000.0
LOOKBACK = 14
VOL_MULT = 1.0
SIGMA_TARGET = 0.02
MAX_LEV = 4.0
COMMISSION_RATE = 0.0001   # 万1/边
SLIPPAGE_TICK = 0.001      # 元/份/边
LOT = 100
SAMPLE_START = pd.Timestamp("2017-01-01")
SAMPLE_END = pd.Timestamp("2025-12-31")


def make_grid(step_min, last_pm):
    am, pm = [], []
    t = 10 * 60
    while t <= 11 * 60:
        am.append(f"{t // 60:02d}:{t % 60:02d}")
        t += step_min
    t = 13 * 60
    last = int(last_pm[:2]) * 60 + int(last_pm[3:])
    while t <= last:
        pm.append(f"{t // 60:02d}:{t % 60:02d}")
        t += step_min
    return am + pm


GRIDS = {"g30": make_grid(30, "14:30"),
         "g15": make_grid(15, "14:45"),
         "g05": make_grid(5, "14:55")}
# END 标签 bar 语义：决策时点 τ 的信号 bar = 标签 τ（13:00 例外，用上午收盘 bar 11:30）；
# 成交 bar = 信号 bar 的下一分钟标签
SIG_BAR = {"13:00": "11:30"}


def label_key(s):
    return int(s[:2]) * 60 + int(s[3:])


def perf_stats(ret, equity, init=INIT_CAPITAL):
    r = np.asarray(ret, dtype=float)
    eq = np.asarray(equity, dtype=float)
    n = len(r)
    total = eq[-1] / init - 1.0
    irr = (1.0 + total) ** (252.0 / n) - 1.0 if (n > 0 and total > -1) else np.nan
    sd = r.std(ddof=1) if n > 1 else np.nan
    vol = sd * math.sqrt(252.0)
    sharpe = r.mean() / sd * math.sqrt(252.0) if (sd and sd > 0) else np.nan
    hit = float((r > 0).mean()) if n else np.nan
    peak = np.maximum.accumulate(eq)
    mdd = float((eq / peak - 1.0).min()) if n else np.nan
    skew = float(pd.Series(r).skew()) if n > 2 else np.nan
    return {"N": n, "TotalReturn": total, "IRR": irr, "Vol": vol, "Sharpe": sharpe,
            "HitRatio": hit, "MDD": mdd, "Skew": skew,
            "WorstDay": float(r.min()) if n else np.nan,
            "BestDay": float(r.max()) if n else np.nan,
            "AvgDailyBps": float(r.mean()) * 1e4 if n else np.nan}


def ols_stats(y, x):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = ~(np.isnan(x) | np.isnan(y))
    x, y = x[m], y[m]
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    s2 = float(resid @ resid) / (n - 2)
    se = np.sqrt(np.diag(s2 * np.linalg.inv(X.T @ X)))
    tvals = beta / se
    pvals = [math.erfc(abs(t) / math.sqrt(2.0)) for t in tvals]
    ss_tot = float(((y - y.mean()) ** 2).sum())
    r2 = 1.0 - float(resid @ resid) / ss_tot if ss_tot > 0 else np.nan
    return {"alpha": float(beta[0]), "beta": float(beta[1]),
            "t_alpha": float(tvals[0]), "t_beta": float(tvals[1]),
            "p_alpha": float(pvals[0]), "p_beta": float(pvals[1]), "n": n, "r2": r2}


def rsi_wilder(close, n=5):
    d = close.diff()
    up = d.clip(lower=0.0)
    dn = -d.clip(upper=0.0)
    au = up.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()
    ad = dn.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()
    rs = au / ad
    return 100.0 - 100.0 / (1.0 + rs)


def dump_json(obj, name):
    p = OUT / name
    with open(p, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    return p


print("BLOCK-00 ok | grids:", {k: (len(v), v[0], v[-1]) for k, v in GRIDS.items()})
print("ROOT =", ROOT)


BLOCK-00 ok | grids: {'g30': (7, '10:00', '14:30'), 'g15': (13, '10:00', '14:45'), 'g05': (37, '10:00', '14:55')}
ROOT = D:\quantitative_project\复现工程\Intraday_Momentum_for_ETF


## BLOCK-01 数据加载与分钟网格验证

加载 DT-001 冻结产物，验证 bar 标签集合（END 标签，241/日）、与日频交叉锚定、登记不完整日。
输出：`01_data_overview.json`、`01_daily_crosscheck.csv`、`01_incomplete_days.csv`。

In [2]:
# BLOCK-01 数据加载与网格验证（510300.SH）
mn = pd.read_parquet(PROC / "minute_bars_510300.parquet")
daily = pd.read_csv(PROC / "daily_510300.csv", parse_dates=["trade_date"])
cal = pd.read_csv(PROC / "trade_calendar.csv", parse_dates=["cal_date"])
idx_daily = pd.read_csv(PROC / "index_daily_000300.csv", parse_dates=["trade_date"])

mn = mn[(mn.trade_date >= SAMPLE_START) & (mn.trade_date <= SAMPLE_END)].copy()
mn = mn.sort_values("trade_time").reset_index(drop=True)

# 标准标签集合（END 标签，时间序）：09:30 竞价 + 09:31..11:30 + 13:01..15:00
LABELS = sorted(mn.hhmm.unique(), key=label_key)
expected = (["09:30"] + [f"09:{m:02d}" for m in range(31, 60)]
            + [f"10:{m:02d}" for m in range(60)] + [f"11:{m:02d}" for m in range(31)]
            + [f"13:{m:02d}" for m in range(1, 60)] + [f"14:{m:02d}" for m in range(60)]
            + ["15:00"])
assert LABELS == expected, f"label set mismatch: {len(LABELS)}"
IDX = {s: i for i, s in enumerate(LABELS)}
PRICE_LABELS = LABELS[1:]           # 09:31..15:00（240 个价格时点）
PPos = {s: i for i, s in enumerate(PRICE_LABELS)}

bars_per_day = mn.groupby("trade_date").size()
incomplete = bars_per_day[bars_per_day != 241]
assert set(incomplete.index.strftime("%Y%m%d")) == {"20170809"} and incomplete.iloc[0] == 240

# 矩阵化：days x labels
days = pd.DatetimeIndex(sorted(mn.trade_date.unique()))
n_days, n_lab = len(days), len(LABELS)

def to_mat(col):
    piv = mn.pivot(index="trade_date", columns="hhmm", values=col)
    piv = piv.reindex(index=days, columns=LABELS)
    return piv.to_numpy(dtype=float)

O_m, H_m, L_m, C_m = to_mat("open"), to_mat("high"), to_mat("low"), to_mat("close")
VOL_m, AMT_m = to_mat("vol"), to_mat("amount")
assert not np.isnan(C_m[:, IDX["15:00"]]).any(), "15:00 close missing"

# 日 Open = 09:30 竞价 bar 开盘；2017-08-09 缺竞价 bar → 当日首根 bar 开盘
i0930 = IDX["09:30"]
first_valid = np.argmax(~np.isnan(O_m), axis=1)
day_open = np.where(np.isnan(O_m[:, i0930]), O_m[np.arange(n_days), first_valid], O_m[:, i0930])

# 与日频交叉锚定
dly = daily.set_index("trade_date").reindex(days)
agg_close = C_m[:, IDX["15:00"]]
d_close = np.abs(agg_close - dly["close"].to_numpy())
d_open = np.abs(day_open - dly["open"].to_numpy())
assert d_close.max() < 0.002 and d_open.max() < 0.002, (d_close.max(), d_open.max())

anchors = []
for d in ["2019-04-03", "2022-09-15", "2025-06-24"]:
    di = days.get_loc(pd.Timestamp(d))
    anchors.append({"date": d, "minute_close": float(agg_close[di]),
                    "daily_close": float(dly["close"].iloc[di]),
                    "abs_diff": float(abs(agg_close[di] - dly["close"].iloc[di]))})
    assert anchors[-1]["abs_diff"] < 0.002

pd.DataFrame({"trade_date": days, "close_minute": agg_close, "open_minute": day_open,
              "close_daily": dly["close"].to_numpy(), "open_daily": dly["open"].to_numpy(),
              "d_close": d_close, "d_open": d_open}).to_csv(OUT / "01_daily_crosscheck.csv", index=False)
pd.DataFrame({"trade_date": incomplete.index, "n_bars": incomplete.values,
              "reason": "missing 09:30 opening-auction bar (source gap, documented in DT-001)"}
             ).to_csv(OUT / "01_incomplete_days.csv", index=False)
overview = {"ts_code": TS_CODE, "days": int(n_days),
            "first_day": str(days[0].date()), "last_day": str(days[-1].date()),
            "minute_rows": int(len(mn)), "labels": n_lab,
            "bars_per_day": {"standard": 241, "exceptions": {"2017-08-09": 240}},
            "close_maxdiff": float(d_close.max()), "open_maxdiff": float(d_open.max()),
            "anchors": anchors}
dump_json(overview, "01_data_overview.json")
print(f"BLOCK-01 ok | days={n_days} labels={n_lab} close_maxdiff={d_close.max():.6f} open_maxdiff={d_open.max():.6f}")
print("anchors:", anchors)


BLOCK-01 ok | days=2186 labels=241 close_maxdiff=0.000000 open_maxdiff=0.000000
anchors: [{'date': '2019-04-03', 'minute_close': 4.015, 'daily_close': 4.015, 'abs_diff': 0.0}, {'date': '2022-09-15', 'minute_close': 4.09, 'daily_close': 4.09, 'abs_diff': 0.0}, {'date': '2025-06-24', 'minute_close': 3.936, 'daily_close': 3.936, 'abs_diff': 0.0}]


## BLOCK-02 日频序列与 σ_SPY,t

日收益（pct_chg/100，含分红口径）→ 14 日样本标准差（ddof=1，显式 shift(1)，开盘前可得）→ B&H 净值。
日频自 2016-11 起，σ_SPY,t 在样本首日即已预热完成。输出：`02_daily_series.csv`。

In [3]:
# BLOCK-02 日频序列 / sigma14 / B&H
dl_full = daily.sort_values("trade_date").reset_index(drop=True)
dl_full["ret"] = dl_full["pct_chg"] / 100.0
dl_full["sigma14"] = dl_full["ret"].rolling(LOOKBACK).std(ddof=1).shift(1)   # t 日开盘前可得
assert dl_full["sigma14"].iloc[:LOOKBACK].isna().all()
assert dl_full["sigma14"].iloc[LOOKBACK:].notna().all()

# 手工复算 3 日 sigma14
for d in ["2017-01-03", "2020-07-06", "2025-06-24"]:
    i = dl_full.index[dl_full.trade_date == pd.Timestamp(d)][0]
    manual = dl_full["ret"].iloc[i - LOOKBACK:i].std(ddof=1)
    assert abs(manual - dl_full["sigma14"].iloc[i]) < 1e-12, (d, manual, dl_full["sigma14"].iloc[i])

s = dl_full.set_index("trade_date")
sample_mask = (s.index >= SAMPLE_START) & (s.index <= SAMPLE_END)
pre_close_v = s["pre_close"].reindex(days).to_numpy()
ret_v = s["ret"].reindex(days).to_numpy()
sigma14_v = s["sigma14"].reindex(days).to_numpy()
assert not np.isnan(pre_close_v).any() and not np.isnan(ret_v).any()
assert not np.isnan(sigma14_v).any(), "sigma14 应已借 2016-11 起日频完成预热"

bh_nav = np.cumprod(1.0 + ret_v)
manual_nav = float((1.0 + pd.Series(ret_v)).prod())
assert abs(bh_nav[-1] - manual_nav) < 1e-9

pd.DataFrame({"date": days, "open": day_open, "close": agg_close, "pre_close": pre_close_v,
              "ret": ret_v, "sigma14": sigma14_v, "bh_nav": bh_nav}
             ).to_csv(OUT / "02_daily_series.csv", index=False)
print(f"BLOCK-02 ok | sigma14 range=[{np.nanmin(sigma14_v):.5f},{np.nanmax(sigma14_v):.5f}] "
      f"bh_nav_end={bh_nav[-1]:.4f}")


BLOCK-02 ok | sigma14 range=[0.00146,0.04568] bh_nav_end=1.6427


## BLOCK-03 噪声带（Noise Area）

$move_{d,\tau}=|Close_{d,\tau}/Open_d-1|$；$\sigma_{t,\tau}$=前 14 个交易日同列均值
（rolling(14).mean().shift(1)，t 日开盘前可得）；
$UB=\max(Open_t,preClose_t)\times(1+VM\cdot\sigma)$，$LB=\min(Open_t,preClose_t)\times(1-VM\cdot\sigma)$。
输出：`03_noise_bands_sample.csv`（全部日 × g30 决策时点 + 15:00）、`03_sigma_shape.csv`。

In [4]:
# BLOCK-03 噪声带
P_m = C_m[:, 1:]                       # days x 240 price labels
MOVE = np.abs(P_m / day_open[:, None] - 1.0)

def make_sigma(lookback):
    return pd.DataFrame(MOVE).rolling(lookback).mean().shift(1).to_numpy()

SIGMA = {LOOKBACK: make_sigma(LOOKBACK)}
assert np.isnan(SIGMA[LOOKBACK][:LOOKBACK]).all()          # 前 14 个样本日无带 → 不交易
assert not np.isnan(SIGMA[LOOKBACK][LOOKBACK:]).any()

# 手工复算 3 个 (d, tau)
for d, tau in [("2017-02-03", "10:00"), ("2021-03-15", "13:30"), ("2025-06-24", "14:30")]:
    di = days.get_loc(pd.Timestamp(d))
    pi_ = PPos[tau]
    manual = np.nanmean(MOVE[di - LOOKBACK:di, pi_])
    assert abs(manual - SIGMA[LOOKBACK][di, pi_]) < 1e-12, (d, tau)

def bands_for(di, vm, lookback=LOOKBACK):
    sg = SIGMA[lookback][di]
    base_up = max(day_open[di], pre_close_v[di])
    base_dn = min(day_open[di], pre_close_v[di])
    return base_up * (1.0 + vm * sg), base_dn * (1.0 - vm * sg)

# 输出：全部日 × (g30 决策时点 + 15:00)
rows = []
out_taus = GRIDS["g30"] + ["15:00"]
for di in range(n_days):
    ub, lb = bands_for(di, VOL_MULT)
    for tau in out_taus:
        pi_ = PPos[tau if tau != "13:00" else "11:30"]
        rows.append((days[di], tau, SIGMA[LOOKBACK][di, pi_], ub[pi_], lb[pi_]))
pd.DataFrame(rows, columns=["date", "tau", "sigma", "UB", "LB"]).to_csv(
    OUT / "03_noise_bands_sample.csv", index=False)
pd.DataFrame({"price_label": PRICE_LABELS,
              "sigma_mean": np.nanmean(SIGMA[LOOKBACK], axis=0)}).to_csv(
    OUT / "03_sigma_shape.csv", index=False)
sm = np.nanmean(SIGMA[LOOKBACK][LOOKBACK:], axis=0)
assert sm[-1] > sm[0], "sigma 应随时点上升"
print(f"BLOCK-03 ok | sigma(09:31)={sm[0]:.4%} sigma(15:00)={sm[-1]:.4%} "
      f"range=[{np.nanmin(SIGMA[LOOKBACK][LOOKBACK:]):.4%},{np.nanmax(SIGMA[LOOKBACK]):.4%}]")


BLOCK-03 ok | sigma(09:31)=0.1409% sigma(15:00)=0.7484% range=[0.0422%,3.0238%]


## BLOCK-04 VWAP

$VWAP_{t,\tau}=\dfrac{\sum_{bar\le\tau} amount}{(\sum_{bar\le\tau} vol)\times 100}$
（amount 元、vol 手 → 份；零成交 bar 不产生增量）。仅用截至 τ 的日内累计，无前视。
输出：`04_vwap_check.csv`。

In [5]:
# BLOCK-04 VWAP（日内累计）
CS_AMT = np.nancumsum(np.where(np.isnan(AMT_m), 0.0, AMT_m), axis=1)
CS_VOL = np.nancumsum(np.where(np.isnan(VOL_m), 0.0, VOL_m), axis=1)
with np.errstate(invalid="ignore", divide="ignore"):
    VWAP = np.where(CS_VOL > 0, CS_AMT / (CS_VOL * 100.0), np.nan)
# 决策时点（>=10:00）累计量必 > 0
assert not np.isnan(VWAP[:, IDX["10:00"]:]).any()

# 手工复算 3 个 (d, tau)
checks = []
for d, tau in [("2019-04-03", "10:30"), ("2022-09-15", "13:30"), ("2025-06-24", "14:30")]:
    di = days.get_loc(pd.Timestamp(d))
    li = IDX[tau]
    manual = AMT_m[di, :li + 1].sum() / (VOL_m[di, :li + 1].sum() * 100.0)
    assert abs(manual - VWAP[di, li]) < 1e-9, (d, tau)
    checks.append({"date": d, "tau": tau, "vwap": float(VWAP[di, li]), "manual": float(manual)})

rng20 = np.random.RandomState(42).choice(n_days, 20, replace=False)
rows = []
for di in sorted(rng20):
    for tau in GRIDS["g30"]:
        li = IDX[SIG_BAR.get(tau, tau)]
        rows.append((days[di], tau, VWAP[di, li],
                     AMT_m[di, :li + 1].sum() / (VOL_m[di, :li + 1].sum() * 100.0)))
pd.DataFrame(rows, columns=["date", "tau", "vwap", "manual"]).to_csv(OUT / "04_vwap_check.csv", index=False)

# 数量级：日内 VWAP(15:00) ≈ 日线 amount/vol 均价
dly_avg = (dly["amount"].to_numpy() * 1000.0) / (dly["vol"].to_numpy() * 100.0)
rel = np.abs(VWAP[:, IDX["15:00"]] / dly_avg - 1.0)
print(f"BLOCK-04 ok | vwap(15:00) vs daily avg price rel diff: median={np.nanmedian(rel):.4%} max={np.nanmax(rel):.4%}")
print("checks:", checks)


BLOCK-04 ok | vwap(15:00) vs daily avg price rel diff: median=0.0012% max=0.0526%
checks: [{'date': '2019-04-03', 'tau': '10:30', 'vwap': 3.958580292136163, 'manual': 3.958580292136163}, {'date': '2022-09-15', 'tau': '13:30', 'vwap': 4.119777147121343, 'manual': 4.119777147121343}, {'date': '2025-06-24', 'tau': '14:30', 'vwap': 3.9319232412610723, 'manual': 3.9319232412610723}]


## BLOCK-05 回测引擎与变体 A（Opposite Band）

引擎规格（plan_code.md BLOCK-05）：
- 决策 τ ∈ grid；信号价 p(τ)=标签 τ bar 收盘（13:00 用 11:30 bar）；成交价=下一分钟 bar 开盘；15:00 强平=15:00 bar 收盘
- 空仓且**新跨越**上带 → 开多；新跨越下带 → 开空（armed 机制：价格回到带内/另一侧后重新武装）
- 变体 A 止损=反向带（持多 p<LB 平多+反手开空；对称）；15:00 强制平仓
- 份额=floor(AUM×lev/Open/100)×100；成本=佣金×成交额+滑点×份额，双边；每网格独立资金线
- 成交前查 tradability：该分钟不可买/不可卖则放弃本次委托

输出：`05_trades_oppband_{grid}.csv`、`05_daily_returns_oppband_{grid}.csv`、`05_equity_oppband_{grid}.csv`、`05_meta.json`。

In [6]:
# BLOCK-05 引擎 + 变体 A（三网格）
trad = pd.read_csv(PROC / "tradability.csv", dtype={"交易日期": str, "交易时间": str})
trad["label"] = trad["交易时间"].str[:2] + ":" + trad["交易时间"].str[2:]
trad["trade_date"] = pd.to_datetime(trad["交易日期"], format="%Y%m%d")
BUYABLE = np.ones((n_days, n_lab), dtype=bool)
SELLABLE = np.ones((n_days, n_lab), dtype=bool)
day_pos = {d: i for i, d in enumerate(days)}
tb = trad[trad["可买"] == 0]
for r in tb.itertuples():
    BUYABLE[day_pos[r.trade_date], IDX[r.label]] = False
ts_ = trad[trad["可卖"] == 0]
for r in ts_.itertuples():
    SELLABLE[day_pos[r.trade_date], IDX[r.label]] = False
# 零成交分钟（含 2024-09-27 上交所技术故障 6 根）不可成交：双边封锁
tz = trad[trad["是否有成交"] == 0]
for r in tz.itertuples():
    BUYABLE[day_pos[r.trade_date], IDX[r.label]] = False
    SELLABLE[day_pos[r.trade_date], IDX[r.label]] = False
print(f"tradability gates: buy-blocked={int((~BUYABLE).sum())} sell-blocked={int((~SELLABLE).sum())} "
      f"(incl. zero-activity minutes={len(tz)}; none at 15:00)")

# 网格规格：(tau, sig_col, exec_col, price_pos)
GRID_SPEC = {}
for g, taus in GRIDS.items():
    spec = []
    for tau in taus:
        sbar = SIG_BAR.get(tau, tau)
        sc = IDX[sbar]
        ec = IDX[LABELS[sc + 1]]          # 信号 bar 的下一分钟标签
        spec.append((tau, sc, ec, PPos[sbar]))
    GRID_SPEC[g] = spec
for g, spec in GRID_SPEC.items():
    for tau, sc, ec, _ in spec:
        assert label_key(LABELS[sc]) <= label_key(LABELS[ec]), (g, tau)

# 引擎热路径用 Python list（np.float64 标量运算慢 3-5 倍）
C_l = C_m.tolist()
O_l = O_m.tolist()
VW_l = VWAP.tolist()
BUY_l = BUYABLE.tolist()
SELL_l = SELLABLE.tolist()
s14_l = sigma14_v.tolist()
do_l = day_open.tolist()
pc_l = pre_close_v.tolist()


def run_backtest(grid, stop_mode, dyn_lev=False, vm=1.0, lookback=LOOKBACK,
                 commission=COMMISSION_RATE, slippage=SLIPPAGE_TICK,
                 sigma_target=SIGMA_TARGET, max_lev=MAX_LEV, tag=""):
    if lookback not in SIGMA:
        SIGMA[lookback] = make_sigma(lookback)
    sig_l = SIGMA[lookback].tolist()
    spec = GRID_SPEC[grid]
    i1500 = IDX["15:00"]
    trades = []
    daily_rows = []
    aum = INIT_CAPITAL
    tid = 0
    isnan = math.isnan
    floor = math.floor
    for di in range(n_days):
        aum0 = aum
        pnl_day = 0.0
        n_tr = 0
        s14 = s14_l[di]
        if dyn_lev:
            if isnan(s14) or s14 <= 0:
                daily_rows.append((days[di], aum0, aum0, 0.0, 0, np.nan, 0))
                continue
            lev = min(max_lev, sigma_target / s14)
        else:
            lev = 1.0
        do = do_l[di]
        pc = pc_l[di]
        shares = floor(aum0 * lev / do / 100.0) * 100
        if shares <= 0:
            daily_rows.append((days[di], aum0, aum0, 0.0, 0, lev, 0))
            continue
        base_up = do if do > pc else pc
        base_dn = do if do < pc else pc
        Crow = C_l[di]
        Orow = O_l[di]
        VWrow = VW_l[di]
        SGrow = sig_l[di]
        BUYrow = BUY_l[di]
        SELLrow = SELL_l[di]
        pos = 0
        entry_px = 0.0
        entry_comm = 0.0
        entry_slip = 0.0
        entry_tau = ""
        entry_tid = 0
        armed_l = True
        armed_s = True
        for tau, sc, ec, pi_ in spec:
            p = Crow[sc]
            if p != p:
                continue
            sg = SGrow[pi_]
            if sg != sg:
                continue
            ub = base_up * (1.0 + vm * sg)
            lb = base_dn * (1.0 - vm * sg)
            if p <= ub:
                armed_l = True
            if p >= lb:
                armed_s = True
            ep = Orow[ec]
            if ep != ep:
                continue
            if pos == 0:
                if p > ub and armed_l and BUYrow[ec]:
                    tid += 1
                    entry_tid = tid
                    entry_comm = ep * shares * commission
                    entry_slip = shares * slippage
                    entry_px = ep
                    entry_tau = tau
                    pos = 1
                    armed_l = False
                    trades.append((days[di], tau, LABELS[ec], "买入", ep, shares,
                                   entry_comm, entry_slip, tid, "OPEN_LONG", "", lev))
                    n_tr += 1
                elif p < lb and armed_s and SELLrow[ec]:
                    tid += 1
                    entry_tid = tid
                    entry_comm = ep * shares * commission
                    entry_slip = shares * slippage
                    entry_px = ep
                    entry_tau = tau
                    pos = -1
                    armed_s = False
                    trades.append((days[di], tau, LABELS[ec], "卖出", ep, shares,
                                   entry_comm, entry_slip, tid, "OPEN_SHORT", "", lev))
                    n_tr += 1
            elif pos == 1:
                vw = VWrow[sc]
                if stop_mode == "opp_band":
                    stop = lb
                elif stop_mode == "vwap_only":
                    stop = vw if vw == vw else lb
                else:
                    stop = max(ub, vw) if vw == vw else ub
                if p < stop and SELLrow[ec]:
                    exit_comm = ep * shares * commission
                    exit_slip = shares * slippage
                    pnl_day += (ep - entry_px) * shares - entry_comm - entry_slip - exit_comm - exit_slip
                    trades.append((days[di], tau, LABELS[ec], "卖出", ep, shares,
                                   exit_comm, exit_slip, entry_tid, "CLOSE_LONG", stop_mode, lev))
                    n_tr += 1
                    pos = 0
                    if p < lb and armed_s and SELLrow[ec]:
                        tid += 1
                        entry_tid = tid
                        entry_comm = ep * shares * commission
                        entry_slip = shares * slippage
                        entry_px = ep
                        entry_tau = tau
                        pos = -1
                        armed_s = False
                        trades.append((days[di], tau, LABELS[ec], "卖出", ep, shares,
                                       entry_comm, entry_slip, tid, "OPEN_SHORT", "REVERSAL", lev))
                        n_tr += 1
            elif pos == -1:
                vw = VWrow[sc]
                if stop_mode == "opp_band":
                    stop = ub
                elif stop_mode == "vwap_only":
                    stop = vw if vw == vw else ub
                else:
                    stop = min(lb, vw) if vw == vw else lb
                if p > stop and BUYrow[ec]:
                    exit_comm = ep * shares * commission
                    exit_slip = shares * slippage
                    pnl_day += (entry_px - ep) * shares - entry_comm - entry_slip - exit_comm - exit_slip
                    trades.append((days[di], tau, LABELS[ec], "买入", ep, shares,
                                   exit_comm, exit_slip, entry_tid, "CLOSE_SHORT", stop_mode, lev))
                    n_tr += 1
                    pos = 0
                    if p > ub and armed_l and BUYrow[ec]:
                        tid += 1
                        entry_tid = tid
                        entry_comm = ep * shares * commission
                        entry_slip = shares * slippage
                        entry_px = ep
                        entry_tau = tau
                        pos = 1
                        armed_l = False
                        trades.append((days[di], tau, LABELS[ec], "买入", ep, shares,
                                       entry_comm, entry_slip, tid, "OPEN_LONG", "REVERSAL", lev))
                        n_tr += 1
        if pos != 0:
            ep = Crow[i1500]
            exit_comm = ep * shares * commission
            exit_slip = shares * slippage
            if pos == 1:
                pnl_day += (ep - entry_px) * shares - entry_comm - entry_slip - exit_comm - exit_slip
                trades.append((days[di], "15:00", "15:00", "卖出", ep, shares,
                               exit_comm, exit_slip, entry_tid, "CLOSE_LONG", "EOD", lev))
            else:
                pnl_day += (entry_px - ep) * shares - entry_comm - entry_slip - exit_comm - exit_slip
                trades.append((days[di], "15:00", "15:00", "买入", ep, shares,
                               exit_comm, exit_slip, entry_tid, "CLOSE_SHORT", "EOD", lev))
            n_tr += 1
        if aum0 + pnl_day <= 0:
            raise RuntimeError(f"equity non-positive at {days[di].date()}")
        aum = aum0 + pnl_day
        daily_rows.append((days[di], aum0, aum, pnl_day / aum0, n_tr, lev, 1 if n_tr > 0 else 0))
    tr = pd.DataFrame(trades, columns=["date", "decision_time", "exec_time", "side", "price",
                                       "shares", "commission", "slippage_cost", "trade_id",
                                       "action", "reason", "lev"])
    tr["cost"] = tr["commission"] + tr["slippage_cost"]
    dl = pd.DataFrame(daily_rows, columns=["date", "aum_start", "aum_end", "ret",
                                           "n_exec", "lev", "traded"])
    eq = dl[["date", "aum_end"]].copy()
    eq["nav"] = eq["aum_end"] / INIT_CAPITAL
    return {"trades": tr, "daily": dl, "equity": eq,
            "params": {"grid": grid, "stop_mode": stop_mode, "dyn_lev": dyn_lev, "vm": vm,
                       "lookback": lookback, "commission": commission, "slippage": slippage,
                       "sigma_target": sigma_target, "max_lev": max_lev, "tag": tag}}


def save_run(res, prefix):
    g = res["params"]["grid"]
    res["trades"].to_csv(OUT / f"{prefix}_trades_{g}.csv", index=False)
    res["daily"].to_csv(OUT / f"{prefix}_daily_returns_{g}.csv", index=False)
    res["equity"].to_csv(OUT / f"{prefix}_equity_{g}.csv", index=False)


RUNS = {}
for g in ["g30", "g15", "g05"]:
    res = run_backtest(g, "opp_band", dyn_lev=False, tag=f"A_{g}")
    RUNS[("A", g)] = res
    save_run(res, "05_oppband")
    # 现金账连续性勾稽：AUM 递推 == sum(pnl)
    rec = res["daily"]
    chk = np.abs(rec["aum_end"].to_numpy() - (INIT_CAPITAL + (rec["ret"] * rec["aum_start"]).cumsum().to_numpy())).max()
    assert chk < 1e-6, chk
    assert (res["equity"]["aum_end"] > 0).all()
    print(f"variant A {g}: exec_rows={len(res['trades'])} days_traded={int(res['daily']['traded'].sum())} "
          f"end_aum={res['daily']['aum_end'].iloc[-1]:,.0f} aum_recursion_maxdiff={chk:.2e}")

dump_json({"variant": "A opp_band", "grids": {g: {"exec_rows": int(len(RUNS[("A", g)]["trades"])),
           "end_aum": float(RUNS[("A", g)]["daily"]["aum_end"].iloc[-1])} for g in GRIDS}}, "05_meta.json")

# 抽 1 笔手工复算（g30 首笔开仓）
t0 = RUNS[("A", "g30")]["trades"].iloc[0]
di0 = day_pos[pd.Timestamp(t0["date"])]
manual_px = O_m[di0, IDX[t0["exec_time"]]]
assert abs(manual_px - t0["price"]) < 1e-9
assert abs(t0["commission"] - t0["price"] * t0["shares"] * COMMISSION_RATE) < 1e-9
assert t0["shares"] % 100 == 0
print("manual trade recheck ok:", t0[["date", "decision_time", "exec_time", "side", "price", "shares"]].to_dict())


tradability gates: buy-blocked=219 sell-blocked=211 (incl. zero-activity minutes=211; none at 15:00)
variant A g30: exec_rows=2840 days_traded=1375 end_aum=1,222,781 aum_recursion_maxdiff=3.73e-09
variant A g15: exec_rows=3068 days_traded=1469 end_aum=1,179,991 aum_recursion_maxdiff=2.33e-09


variant A g05: exec_rows=3332 days_traded=1575 end_aum=1,251,403 aum_recursion_maxdiff=4.19e-09
manual trade recheck ok: {'date': Timestamp('2017-01-23 00:00:00'), 'decision_time': '10:00', 'exec_time': '10:01', 'side': '买入', 'price': 3.378, 'shares': 297600}


## BLOCK-06 变体 B（Current Band + VWAP）

持多止损 = max(UB, VWAP)，持空止损 = min(LB, VWAP)；价格同时穿越反向带则平仓+反手；
止损后不原地再入（armed 机制需新跨越）。输出：`06_*_{grid}.csv`、`06_meta.json`。

In [7]:
# BLOCK-06 变体 B（三网格）
for g in ["g30", "g15", "g05"]:
    res = run_backtest(g, "band_vwap", dyn_lev=False, tag=f"B_{g}")
    RUNS[("B", g)] = res
    save_run(res, "06_bandvwap")
    nA = len(RUNS[("A", g)]["trades"])
    nB = len(res["trades"])
    assert (res["equity"]["aum_end"] > 0).all()
    print(f"variant B {g}: exec_rows={nB} (A={nA}, B/A={nB / nA:.2f}) "
          f"end_aum={res['daily']['aum_end'].iloc[-1]:,.0f}")

# 抽 3 个止损案例核对 stop = max(UB, VWAP)
tb30 = RUNS[("B", "g30")]["trades"]
cl = tb30[tb30.action == "CLOSE_LONG"].head(3)
for r in cl.itertuples():
    di = day_pos[pd.Timestamp(r.date)]
    sc = IDX[SIG_BAR.get(r.decision_time, r.decision_time)]
    pi_ = PPos[SIG_BAR.get(r.decision_time, r.decision_time)]
    sg = SIGMA[LOOKBACK][di, pi_]
    ub = max(day_open[di], pre_close_v[di]) * (1 + sg)
    vw = VWAP[di, sc]
    p = C_m[di, sc]
    assert p < max(ub, vw), (r.date, r.decision_time)
dump_json({"variant": "B band_vwap",
           "grids": {g: {"exec_rows": int(len(RUNS[("B", g)]["trades"])),
                         "end_aum": float(RUNS[("B", g)]["daily"]["aum_end"].iloc[-1])} for g in GRIDS}},
          "06_meta.json")
print("BLOCK-06 ok | 3 stop cases verified (p < max(UB,VWAP))")


variant B g30: exec_rows=3410 (A=2840, B/A=1.20) end_aum=834,476
variant B g15: exec_rows=4318 (A=3068, B/A=1.41) end_aum=661,381
variant B g05: exec_rows=6430 (A=3332, B/A=1.93) end_aum=367,069


BLOCK-06 ok | 3 stop cases verified (p < max(UB,VWAP))


## BLOCK-07 变体 C（B + 波动率目标动态仓位）

$lev_t=\min(4,\ \sigma_{target}/\sigma_{SPY,t})$，$\sigma_{target}=2\%$；
份额=floor(AUM×lev/Open/100)×100；止损同变体 B。输出：`07_*_{grid}.csv`、`07_leverage.csv`、`07_meta.json`。

In [8]:
# BLOCK-07 变体 C（三网格）
for g in ["g30", "g15", "g05"]:
    res = run_backtest(g, "band_vwap", dyn_lev=True, tag=f"C_{g}")
    RUNS[("C", g)] = res
    save_run(res, "07_dynamic")
    lev_v = res["daily"]["lev"].dropna()
    assert (lev_v > 0).all() and (lev_v <= MAX_LEV + 1e-12).all()
    assert (res["equity"]["aum_end"] > 0).all()
    print(f"variant C {g}: exec_rows={len(res['trades'])} end_aum={res['daily']['aum_end'].iloc[-1]:,.0f} "
          f"lev mean={lev_v.mean():.2f} max={lev_v.max():.2f}")

# 抽 3 日手工复算 lev 与份额
rc = RUNS[("C", "g30")]["daily"]
for d in ["2018-06-11", "2021-09-01", "2025-06-24"]:
    row = rc[rc.date == pd.Timestamp(d)].iloc[0]
    di = day_pos[pd.Timestamp(d)]
    lev_manual = min(MAX_LEV, SIGMA_TARGET / sigma14_v[di])
    assert abs(row.lev - lev_manual) < 1e-12, (d, row.lev, lev_manual)
    sh_manual = math.floor(row.aum_start * lev_manual / day_open[di] / 100.0) * 100
    tr_d = RUNS[("C", "g30")]["trades"]
    td = tr_d[tr_d.date == pd.Timestamp(d)]
    if len(td):
        assert td["shares"].iloc[0] == sh_manual, (d, td["shares"].iloc[0], sh_manual)

lev_all = rc[["date", "lev", "aum_start"]].copy()
lev_all["notional"] = lev_all["lev"] * lev_all["aum_start"]
lev_all.to_csv(OUT / "07_leverage.csv", index=False)
dump_json({"variant": "C dynamic",
           "grids": {g: {"exec_rows": int(len(RUNS[("C", g)]["trades"])),
                         "end_aum": float(RUNS[("C", g)]["daily"]["aum_end"].iloc[-1]),
                         "lev_mean": float(RUNS[("C", g)]["daily"]["lev"].mean()),
                         "lev_max": float(RUNS[("C", g)]["daily"]["lev"].max())} for g in GRIDS}},
          "07_meta.json")
print("BLOCK-07 ok | lev/shares manual recheck passed")


variant C g30: exec_rows=3410 end_aum=427,347 lev mean=2.20 max=4.00


variant C g15: exec_rows=4318 end_aum=260,057 lev mean=2.20 max=4.00


variant C g05: exec_rows=6430 end_aum=58,622 lev mean=2.20 max=4.00


BLOCK-07 ok | lev/shares manual recheck passed


## BLOCK-08 汇总统计（Tables 1/2/3/A1）+ Alpha/Beta

Total Return、IRR（(1+TR)^(252/N)−1）、Vol、Sharpe（rf=0）、Hit Ratio（全样本日 + traded-days 两口径）、
MDD、Skew、Worst/Best Day；日频 OLS：RetMom = α + β·RetETF + ε（α 年化 ×252）。
输出：`08_summary_stats.csv`（变体×网格 + B&H）、`08_alpha_beta.csv`。

In [9]:
# BLOCK-08 汇总统计 + alpha/beta
rows = []
ab_rows = []
for (v, g), res in RUNS.items():
    dl = res["daily"]
    st = perf_stats(dl["ret"], dl["aum_end"])
    traded = dl[dl["traded"] == 1]
    st["HitRatioTraded"] = float((traded["ret"] > 0).mean()) if len(traded) else np.nan
    st["TradedDays"] = int(len(traded))
    st["Variant"] = v
    st["Grid"] = g
    rows.append(st)
    ab = ols_stats(dl["ret"].to_numpy(), ret_v)
    ab_rows.append({"Variant": v, "Grid": g, "alpha_daily": ab["alpha"],
                    "alpha_annual": ab["alpha"] * 252.0, "beta": ab["beta"],
                    "t_alpha": ab["t_alpha"], "t_beta": ab["t_beta"],
                    "p_alpha": ab["p_alpha"], "p_beta": ab["p_beta"], "n": ab["n"], "r2": ab["r2"]})
st = perf_stats(ret_v, bh_nav * INIT_CAPITAL)
st.update({"HitRatioTraded": np.nan, "TradedDays": int((ret_v != 0).sum()),
           "Variant": "BH", "Grid": "-"})
rows.append(st)

SUMMARY = pd.DataFrame(rows)[["Variant", "Grid", "N", "TradedDays", "TotalReturn", "IRR", "Vol",
                              "Sharpe", "HitRatio", "HitRatioTraded", "MDD", "Skew",
                              "WorstDay", "BestDay", "AvgDailyBps"]]
SUMMARY.to_csv(OUT / "08_summary_stats.csv", index=False)
ALPHA_BETA = pd.DataFrame(ab_rows)
ALPHA_BETA.to_csv(OUT / "08_alpha_beta.csv", index=False)

# 勾稽：TotalReturn == equity 末值/初值 - 1
for (v, g), res in RUNS.items():
    tr_direct = res["equity"]["nav"].iloc[-1] - 1.0
    got = SUMMARY[(SUMMARY.Variant == v) & (SUMMARY.Grid == g)]["TotalReturn"].iloc[0]
    assert abs(tr_direct - got) < 1e-12, (v, g)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("BLOCK-08 ok")
print(SUMMARY[SUMMARY.Grid.isin(["g30", "-"])].to_string(index=False))


BLOCK-08 ok
Variant Grid    N  TradedDays  TotalReturn     IRR    Vol  Sharpe  HitRatio  HitRatioTraded     MDD    Skew  WorstDay  BestDay  AvgDailyBps
      A  g30 2186        1375       0.2228  0.0235 0.1078  0.2689    0.3038          0.4829 -0.2329  0.6547   -0.0366   0.0475       1.1498
      B  g30 2186        1375      -0.1655 -0.0206 0.0881 -0.1931    0.2274          0.3615 -0.2921  2.0782   -0.0299   0.0475      -0.6750
      C  g30 2186        1375      -0.5727 -0.0934 0.1696 -0.4941    0.2274          0.3615 -0.5727  1.6656   -0.0460   0.0829      -3.3245
     BH    - 2186        2161       0.6427  0.0589 0.1892  0.3970    0.5041             NaN -0.4216 -0.0041   -0.0884   0.0941       2.9813


## BLOCK-09 交易级统计（Table 4）

trade = 整趟持仓（开→平）；PnL/Share = 整趟净盈亏/份额（含成本）；Max Loss/Gain 按整趟收益率。
输出：`09_trade_level_stats.csv`、`09_round_trips_{variant}_{grid}.csv`。

In [10]:
# BLOCK-09 交易级统计
def round_trips(tr):
    rts = []
    for tid, grp in tr.groupby("trade_id"):
        grp = grp.sort_index()
        op = grp[grp.action.str.startswith("OPEN")]
        cl = grp[grp.action.str.startswith("CLOSE")]
        if len(op) != 1 or len(cl) < 1:
            raise RuntimeError(f"bad round trip {tid}")
        o = op.iloc[0]
        c = cl.iloc[-1]
        direction = 1 if o.action == "OPEN_LONG" else -1
        gross = direction * (c["price"] - o["price"]) * o["shares"]
        costs = grp["cost"].sum()
        pnl = gross - costs
        rts.append({"trade_id": tid, "date": o["date"], "direction": "long" if direction == 1 else "short",
                    "entry_time": o["decision_time"], "exit_time": c["decision_time"],
                    "entry_px": o["price"], "exit_px": c["price"], "shares": o["shares"],
                    "gross": gross, "costs": costs, "pnl": pnl,
                    "pnl_per_share": pnl / o["shares"],
                    "ret_on_notional": pnl / (o["price"] * o["shares"]),
                    "n_exec": len(grp)})
    return pd.DataFrame(rts)


RT = {}
rows = []
for (v, g), res in RUNS.items():
    rt = round_trips(res["trades"])
    RT[(v, g)] = rt
    rt.to_csv(OUT / f"09_round_trips_{v}_{g}.csv", index=False)
    ndays = len(res["daily"])
    worst = rt.loc[rt["ret_on_notional"].idxmin()]
    best = rt.loc[rt["ret_on_notional"].idxmax()]
    rows.append({"Variant": v, "Grid": g, "Trades": len(rt),
                 "AvgTradePerDay": len(rt) / ndays,
                 "HitRatio": float((rt["pnl"] > 0).mean()),
                 "AvgPnLPerShare": float(rt["pnl_per_share"].mean()),
                 "AvgRetOnNotionalBps": float(rt["ret_on_notional"].mean()) * 1e4,
                 "MaxLossTradePct": float(worst["ret_on_notional"]) * 100,
                 "MaxLossDate": str(pd.Timestamp(worst["date"]).date()),
                 "MaxGainTradePct": float(best["ret_on_notional"]) * 100,
                 "MaxGainDate": str(pd.Timestamp(best["date"]).date()),
                 "LongShare": float((rt["direction"] == "long").mean())})
TLS = pd.DataFrame(rows)
TLS.to_csv(OUT / "09_trade_level_stats.csv", index=False)
# 勾稽：整趟数 = trades 文件 OPEN 行数；hit 与符号一致；日盈亏合计 == 整趟 pnl 合计
for (v, g), res in RUNS.items():
    n_open = int(res["trades"]["action"].str.startswith("OPEN").sum())
    assert n_open == len(RT[(v, g)]), (v, g)
    pnl_sum_rt = RT[(v, g)]["pnl"].sum()
    pnl_sum_daily = (res["daily"]["ret"] * res["daily"]["aum_start"]).sum()
    assert abs(pnl_sum_rt - pnl_sum_daily) < 1e-6, (v, g, pnl_sum_rt, pnl_sum_daily)
print("BLOCK-09 ok")
print(TLS[TLS.Grid == "g30"].to_string(index=False))


BLOCK-09 ok
Variant Grid  Trades  AvgTradePerDay  HitRatio  AvgPnLPerShare  AvgRetOnNotionalBps  MaxLossTradePct MaxLossDate  MaxGainTradePct MaxGainDate  LongShare
      A  g30    1420          0.6496    0.4810          0.0009               1.7485          -3.4009  2020-03-18           4.7072  2024-09-30     0.5134
      B  g30    1705          0.7800    0.3472         -0.0004              -0.8898          -1.9564  2022-10-25           4.7072  2024-09-30     0.5067
      C  g30    1705          0.7800    0.3472         -0.0004              -0.8898          -1.9564  2022-10-25           4.7072  2024-09-30     0.5067


## BLOCK-10 图形（Fig 1-7）

Fig1 噪声带概念；Fig2 案例日（2022-01-31 多 / 2022-04-29 空，510300 同日历日复刻）；
Fig3 权益 A vs B&H；Fig4/Fig5 案例 2022-01-20（变体 A vs B 止损对比）；Fig6 权益 A/B vs B&H；Fig7 权益 A/B/C vs B&H。
输出：`10_fig*.png` 及 `10_*_data.csv`。

In [11]:
# BLOCK-10 图形
XK = np.array([label_key(s) for s in PRICE_LABELS]) / 60.0   # 小时

def day_bands_at_taus(di, taus, vm=VOL_MULT, lookback=LOOKBACK):
    ub, lb = bands_for(di, vm, lookback)
    xs, u, l = [], [], []
    for tau in taus:
        s = SIG_BAR.get(tau, tau)
        xs.append(label_key(s) / 60.0)
        u.append(ub[PPos[s]])
        l.append(lb[PPos[s]])
    return np.array(xs), np.array(u), np.array(l)


def plot_day(ax, di, title, trades=None, show_vwap=True):
    ax.plot(XK, C_m[di, 1:], color="black", lw=1.2, label="510300 price")
    xb, ub, lb = day_bands_at_taus(di, GRIDS["g30"])
    ax.step(xb, ub, where="post", color="tab:red", lw=1.0, label="UB")
    ax.step(xb, lb, where="post", color="tab:green", lw=1.0, label="LB")
    ax.fill_between(xb, lb, ub, step="post", alpha=0.15, color="gray", label="Noise area")
    if show_vwap:
        ax.plot(XK, VWAP[di, 1:], color="tab:blue", lw=0.9, ls="--", label="VWAP")
    if trades is not None and len(trades):
        for r in trades.itertuples():
            x = label_key(r.exec_time) / 60.0
            mk = "^" if r.side == "买入" else "v"
            c = "tab:red" if r.side == "买入" else "tab:green"
            ax.scatter([x], [r.price], marker=mk, color=c, s=55, zorder=5)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Hour of day")
    ax.legend(fontsize=7, loc="best")
    ax.grid(alpha=0.25)


def nearest_di(d):
    ts = pd.Timestamp(d)
    if ts in day_pos:
        return day_pos[ts], d
    nxt = days[days.searchsorted(ts)]
    return day_pos[nxt], f"{d}(A-share closed) -> {nxt.date()}"


# Fig 1 概念图（2022-01-31；A 股春节休市 → 顺延至 2022-02-07）
di_f1, lab_f1 = nearest_di("2022-01-31")
fig, ax = plt.subplots(figsize=(9, 4.5))
plot_day(ax, di_f1, f"Fig1 Noise-area concept (510300, {lab_f1})")
fig.tight_layout()
fig.savefig(OUT / "10_fig1_concept.png")
plt.close(fig)
pd.DataFrame({"hour": XK, "close": C_m[di_f1, 1:], "vwap": VWAP[di_f1, 1:]}).to_csv(
    OUT / "10_fig1_data.csv", index=False)

# Fig 2 两个案例日（变体 A g30 成交标记；同日历日复刻，遇 A 股休市顺延）
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, d in zip(axes, ["2022-01-31", "2022-04-29"]):
    di, lab = nearest_di(d)
    tr_d = RUNS[("A", "g30")]["trades"]
    plot_day(ax, di, f"Fig2 case {lab} (variant A trades)",
             trades=tr_d[tr_d.date == days[di]])
fig.tight_layout()
fig.savefig(OUT / "10_fig2_case_days.png")
plt.close(fig)

# Fig 3 权益 A vs B&H
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(days, RUNS[("A", "g30")]["equity"]["nav"], label="Variant A (opp band)", lw=1.2)
ax.plot(days, bh_nav, label="B&H 510300", lw=1.0)
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Fig3 Equity: Variant A vs B&H (log scale, g30)")
fig.tight_layout()
fig.savefig(OUT / "10_fig3_equity_oppband.png")
plt.close(fig)
pd.DataFrame({"date": days, "nav_A": RUNS[("A", "g30")]["equity"]["nav"],
              "nav_BH": bh_nav}).to_csv(OUT / "10_fig3_data.csv", index=False)

# Fig 4/5 案例 2022-01-20：变体 A vs 变体 B 止损
di_f4 = day_pos[pd.Timestamp("2022-01-20")]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_day(axes[0], di_f4, "Fig4 2022-01-20 variant A (opp-band stop)",
         trades=RUNS[("A", "g30")]["trades"][lambda x: x.date == pd.Timestamp("2022-01-20")])
plot_day(axes[1], di_f4, "Fig5 2022-01-20 variant B (max(UB,VWAP) stop)",
         trades=RUNS[("B", "g30")]["trades"][lambda x: x.date == pd.Timestamp("2022-01-20")])
fig.tight_layout()
fig.savefig(OUT / "10_fig4_case_20220120.png")
plt.close(fig)
fig, ax = plt.subplots(figsize=(9, 4.5))
plot_day(ax, di_f4, "Fig5 stop comparison 2022-01-20 (A red marks / B blue marks)")
trA = RUNS[("A", "g30")]["trades"]
trB = RUNS[("B", "g30")]["trades"]
for r in trA[trA.date == pd.Timestamp("2022-01-20")].itertuples():
    ax.scatter([label_key(r.exec_time) / 60.0], [r.price], marker="o", color="tab:red", s=45, zorder=5)
for r in trB[trB.date == pd.Timestamp("2022-01-20")].itertuples():
    ax.scatter([label_key(r.exec_time) / 60.0], [r.price], marker="x", color="tab:blue", s=45, zorder=6)
fig.tight_layout()
fig.savefig(OUT / "10_fig5_stops_20220120.png")
plt.close(fig)
pd.DataFrame({"hour": XK, "close": C_m[di_f4, 1:], "vwap": VWAP[di_f4, 1:]}).to_csv(
    OUT / "10_fig4_data.csv", index=False)

# Fig 6 权益 A/B vs B&H
fig, ax = plt.subplots(figsize=(9, 4.5))
for v, c in [("A", "tab:orange"), ("B", "tab:blue")]:
    ax.plot(days, RUNS[(v, "g30")]["equity"]["nav"], label=f"Variant {v}", lw=1.1, color=c)
ax.plot(days, bh_nav, label="B&H 510300", lw=1.0, color="black")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Fig6 Equity: Variants A, B vs B&H (log, g30)")
fig.tight_layout()
fig.savefig(OUT / "10_fig6_equity_ab.png")
plt.close(fig)
pd.DataFrame({"date": days, "nav_A": RUNS[("A", "g30")]["equity"]["nav"],
              "nav_B": RUNS[("B", "g30")]["equity"]["nav"], "nav_BH": bh_nav}
             ).to_csv(OUT / "10_fig6_data.csv", index=False)

# Fig 7 权益 A/B/C vs B&H
fig, ax = plt.subplots(figsize=(9, 4.5))
for v, c in [("A", "tab:orange"), ("B", "tab:blue"), ("C", "tab:red")]:
    ax.plot(days, RUNS[(v, "g30")]["equity"]["nav"], label=f"Variant {v}", lw=1.1, color=c)
ax.plot(days, bh_nav, label="B&H 510300", lw=1.0, color="black")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Fig7 Equity: Variants A, B, C vs B&H (log, g30)")
fig.tight_layout()
fig.savefig(OUT / "10_fig7_equity_abc.png")
plt.close(fig)
pd.DataFrame({"date": days, "nav_A": RUNS[("A", "g30")]["equity"]["nav"],
              "nav_B": RUNS[("B", "g30")]["equity"]["nav"],
              "nav_C": RUNS[("C", "g30")]["equity"]["nav"], "nav_BH": bh_nav}
             ).to_csv(OUT / "10_fig7_data.csv", index=False)
print("BLOCK-10 ok | figures 1-7 saved")


BLOCK-10 ok | figures 1-7 saved


## BLOCK-11 Figure 8 波动机制（σ_SPY 代理）

无 VIX 数据（DEC-006）：以 σ_SPY,t（14 日日收益 std，开盘前可得）分档，
各门槛以上交易日的变体 C（g30）日收益 Sharpe。输出：`11_vol_regime_sharpe.csv`、`11_fig8_vol_regime.png`。

In [12]:
# BLOCK-11 波动机制
dlC = RUNS[("C", "g30")]["daily"]
thr_list = [0.0, 0.005, 0.0075, 0.01, 0.0125, 0.015, 0.02, 0.025, 0.03]
rows = []
for thr in thr_list:
    m = (dlC["lev"].notna()) & (dlC["traded"] == 1) & (pd.Series(sigma14_v, index=days).reindex(dlC.date).to_numpy() >= thr)
    r = dlC.loc[m, "ret"].to_numpy()
    sd = r.std(ddof=1) if len(r) > 1 else np.nan
    rows.append({"sigma_threshold": thr, "days": int(m.sum()),
                 "sharpe": r.mean() / sd * math.sqrt(252.0) if (sd and sd > 0) else np.nan,
                 "avg_bps": float(r.mean()) * 1e4 if len(r) else np.nan})
VR = pd.DataFrame(rows)
VR.to_csv(OUT / "11_vol_regime_sharpe.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(VR["sigma_threshold"] * 100, VR["sharpe"], marker="o")
ax.set_xlabel("sigma_SPY,t threshold (%)")
ax.set_ylabel("Sharpe (traded days above threshold)")
ax.set_title("Fig8 Volatility regime: Sharpe by sigma_SPY threshold (variant C, g30)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "11_fig8_vol_regime.png")
plt.close(fig)
print("BLOCK-11 ok")
print(VR.to_string(index=False))


BLOCK-11 ok
 sigma_threshold  days  sharpe  avg_bps
          0.0000  1375 -0.6231  -5.2853
          0.0050  1280 -0.2596  -2.2150
          0.0075  1000 -0.2629  -2.1252
          0.0100   575  0.6364   5.1977
          0.0125   293  1.3739  11.2391
          0.0150   173  0.3698   2.3695
          0.0200    79  0.7886   4.9043
          0.0250    23  0.0823   0.3489
          0.0300     5 -4.4218 -14.5753


## BLOCK-12 Table 5 八种日内形态（t−1 判定，无前视）

形态由 t−1 及以前日频 OHLC 判定，统计 t 日变体 C（g30）traded-day 收益。
输出：`12_daily_patterns.csv`。

In [13]:
# BLOCK-12 日内形态
H_d = s["high"].reindex(days).to_numpy()
L_d = s["low"].reindex(days).to_numpy()
O_d = s["open"].reindex(days).to_numpy()
C_d = s["close"].reindex(days).to_numpy()
rng = pd.Series(H_d - L_d, index=days)
h_s = pd.Series(H_d, index=days)
l_s = pd.Series(L_d, index=days)
o_s = pd.Series(O_d, index=days)
c_s = pd.Series(C_d, index=days)

is_nr4 = (rng == rng.rolling(4).min())
is_nr7 = (rng == rng.rolling(7).min())
is_id = (h_s < h_s.shift(1)) & (l_s > l_s.shift(1))
is_od = (h_s > h_s.shift(1)) & (l_s < l_s.shift(1))
is_tri = (h_s < h_s.shift(1)) & (h_s < h_s.shift(2)) & (l_s > l_s.shift(1)) & (l_s > l_s.shift(2))
pos_o = (o_s - l_s) / rng.replace(0, np.nan)
pos_c = (c_s - l_s) / rng.replace(0, np.nan)
rng_ok = rng > rng.rolling(14).mean()
is_trend = ((pos_o <= 0.15) & (pos_c >= 0.85) & rng_ok) | ((pos_o >= 0.85) & (pos_c <= 0.15) & rng_ok)
is_bigtail = ((pos_o >= 0.75) & (pos_c >= 0.75)) | ((pos_o <= 0.25) & (pos_c <= 0.25))
is_strong = pos_c >= 0.90
is_weak = pos_c <= 0.10

PATTERNS = {"NR4": is_nr4, "NR7": is_nr7, "ID": is_id, "OD": is_od, "Triangle": is_tri,
            "TrendDay": is_trend, "BigTail": is_bigtail, "StrongClose": is_strong, "WeakClose": is_weak}

# 手工验证 2 日 NR4 / ID
for d in ["2020-03-05", "2023-08-28"]:
    i = days.get_loc(pd.Timestamp(d))
    manual_nr4 = (rng.iloc[i - 1] == rng.iloc[i - 4:i].min())
    assert bool(is_nr4.shift(1).iloc[i]) == bool(manual_nr4), (d, "NR4")
    manual_id = (H_d[i - 1] < H_d[i - 2]) and (L_d[i - 1] > L_d[i - 2])
    assert bool(is_id.shift(1).iloc[i]) == manual_id, (d, "ID")

retC = pd.Series(dlC["ret"].to_numpy(), index=days)
traded = pd.Series(dlC["traded"].to_numpy(), index=days) == 1
rows = []
def pat_stats(mask, name):
    m = mask & traded & retC.notna()
    r = retC[m].to_numpy()
    n = len(r)
    sd = r.std(ddof=1) if n > 1 else np.nan
    rows.append({"Pattern": name, "Observations": n,
                 "AvgPnLBps": float(r.mean()) * 1e4 if n else np.nan,
                 "tstat": float(r.mean() / (sd / math.sqrt(n))) if (sd and sd > 0) else np.nan,
                 "HitRatio": float((r > 0).mean()) if n else np.nan,
                 "Sharpe": float(r.mean() / sd * math.sqrt(252.0)) if (sd and sd > 0) else np.nan})

pat_stats(pd.Series(True, index=days), "Unconditional")
for name, ser in PATTERNS.items():
    pat_stats(ser.shift(1).fillna(False).astype(bool), name)
PAT = pd.DataFrame(rows)
PAT.to_csv(OUT / "12_daily_patterns.csv", index=False)
print("BLOCK-12 ok")
print(PAT.to_string(index=False))


BLOCK-12 ok
      Pattern  Observations  AvgPnLBps   tstat  HitRatio  Sharpe
Unconditional          1375    -5.2853 -1.4555    0.3615 -0.6231
          NR4           351   -21.0274 -3.4413    0.3305 -2.9159
          NR7           210   -14.4224 -1.7844    0.3381 -1.9547
           ID           189   -10.2818 -1.1831    0.3651 -1.3661
           OD           133    -1.6501 -0.1433    0.3684 -0.1972
     Triangle            25   -13.3579 -0.6450    0.3600 -2.0477
     TrendDay           144   -26.9314 -2.8191    0.2778 -3.7294
      BigTail            51     9.8961  0.5343    0.5294  1.1876
  StrongClose           192     6.5148  0.6404    0.3958  0.7336
    WeakClose           165     1.2202  0.1013    0.3455  0.1252


## BLOCK-13 Table 6 星期效应

变体 C（g30）traded-day 日收益按星期分组。输出：`13_day_of_week.csv`。

In [14]:
# BLOCK-13 星期效应
dow = pd.Series(days.dayofweek, index=days)
rows = []
for k, name in enumerate(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]):
    m = (dow == k) & traded & retC.notna()
    r = retC[m].to_numpy()
    n = len(r)
    sd = r.std(ddof=1) if n > 1 else np.nan
    rows.append({"Weekday": name, "Observations": n,
                 "AvgPnLBps": float(r.mean()) * 1e4 if n else np.nan,
                 "tstat": float(r.mean() / (sd / math.sqrt(n))) if (sd and sd > 0) else np.nan,
                 "HitRatio": float((r > 0).mean()) if n else np.nan,
                 "Sharpe": float(r.mean() / sd * math.sqrt(252.0)) if (sd and sd > 0) else np.nan})
DOW = pd.DataFrame(rows)
assert DOW["Observations"].sum() == int(traded.sum())
DOW.to_csv(OUT / "13_day_of_week.csv", index=False)
print("BLOCK-13 ok")
print(DOW.to_string(index=False))


BLOCK-13 ok
  Weekday  Observations  AvgPnLBps   tstat  HitRatio  Sharpe
   Monday           298    -0.2244 -0.0265    0.3255 -0.0243
  Tuesday           271    -0.2318 -0.0269    0.3875 -0.0259
Wednesday           250   -19.9900 -2.7311    0.3400 -2.7420
 Thursday           273   -15.6988 -2.2046    0.3480 -2.1181
   Friday           283     7.5819  0.8850    0.4064  0.8351


## BLOCK-14 Figure 9 波动乘数 VM 扫描

VM ∈ {0.2, 0.4, …, 2.4} 共 12 档，每档全量重跑变体 C（g30）。VM=1 与 BLOCK-08 锚点勾稽。
输出：`14_vm_sweep.csv`、`14_fig9_vm_sweep.png`。

In [15]:
# BLOCK-14 VM 扫描
vm_list = [round(0.2 * k, 1) for k in range(1, 13)]
rows = []
for vm in vm_list:
    res = run_backtest("g30", "band_vwap", dyn_lev=True, vm=vm, tag=f"VM{vm}")
    st = perf_stats(res["daily"]["ret"], res["daily"]["aum_end"])
    rows.append({"VM": vm, "Sharpe": st["Sharpe"], "TotalReturn": st["TotalReturn"],
                 "IRR": st["IRR"], "MDD": st["MDD"], "N_Exec": len(res["trades"])})
VM_SW = pd.DataFrame(rows)
anchor = VM_SW[np.isclose(VM_SW.VM, 1.0)].iloc[0]
base = SUMMARY[(SUMMARY.Variant == "C") & (SUMMARY.Grid == "g30")].iloc[0]
assert abs(anchor["TotalReturn"] - base["TotalReturn"]) < 1e-9, (anchor["TotalReturn"], base["TotalReturn"])
VM_SW.to_csv(OUT / "14_vm_sweep.csv", index=False)
fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(VM_SW))
ax.bar(x - 0.2, VM_SW["Sharpe"], width=0.4, label="Sharpe")
ax2 = ax.twinx()
ax2.bar(x + 0.2, VM_SW["TotalReturn"], width=0.4, color="tab:orange", label="Total Return")
ax.set_xticks(x)
ax.set_xticklabels(VM_SW["VM"])
ax.set_xlabel("Volatility Multiplier (VM)")
ax.set_title("Fig9 VM sweep (variant C, g30)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "14_fig9_vm_sweep.png")
plt.close(fig)
print("BLOCK-14 ok | VM=1 anchor matched")
print(VM_SW.to_string(index=False))


BLOCK-14 ok | VM=1 anchor matched
    VM  Sharpe  TotalReturn     IRR     MDD  N_Exec
0.2000 -1.5394      -0.9561 -0.3025 -0.9563    6290
0.4000 -1.1288      -0.8883 -0.2233 -0.8932    5526
0.6000 -0.8557      -0.7996 -0.1692 -0.8225    4792
0.8000 -0.7565      -0.7411 -0.1442 -0.7623    4124
1.0000 -0.4941      -0.5727 -0.0934 -0.5727    3410
1.2000 -0.2768      -0.3869 -0.0548 -0.4667    2908
1.4000  0.0508      -0.0258 -0.0030 -0.3286    2372
1.6000  0.1758       0.1364  0.0149 -0.3125    2002
1.8000  0.3973       0.4465  0.0435 -0.2702    1658
2.0000  0.4538       0.4814  0.0463 -0.2006    1360
2.2000  0.4730       0.4642  0.0449 -0.1650    1118
2.4000  0.4358       0.3759  0.0375 -0.1560     926


## BLOCK-15 §4.5 RSI(5) 回归

$RetMom_t(bps)=\alpha+\beta\cdot RSI_{SPY,t-1}+\varepsilon$；RSI 用 510300 日收盘，
**Wilder 平滑**（ewm alpha=1/5, adjust=False, min_periods=5）；自变量 t−1 收盘可得；
因变量变体 C（g30）日收益 bps（全样本日，无交易=0）。论文锚点 β=−3.25, p=0.001（SPY 口径）。
输出：`15_rsi_regression.csv`。

In [16]:
# BLOCK-15 RSI 回归
close_full = dl_full.set_index("trade_date")["close"]
rsi_full = rsi_wilder(close_full, 5)
rsi_sample = rsi_full.reindex(days)
# 手工复算 3 日 RSI（Wilder）
for d in ["2020-07-06", "2023-04-10", "2025-06-24"]:
    i = dl_full.index[dl_full.trade_date == pd.Timestamp(d)][0]
    seg = close_full.iloc[:i + 1]
    dlt = seg.diff()
    up = dlt.clip(lower=0.0)
    dn = -dlt.clip(upper=0.0)
    au = up.iloc[1:6].mean()
    ad = dn.iloc[1:6].mean()
    for k in range(6, len(seg)):
        au = (au * 4 + up.iloc[k]) / 5.0
        ad = (ad * 4 + dn.iloc[k]) / 5.0
    manual = 100.0 - 100.0 / (1.0 + au / ad) if ad > 0 else 100.0
    assert abs(manual - rsi_full.iloc[i]) < 1e-9, (d, manual, rsi_full.iloc[i])

y_bps = retC.to_numpy() * 1e4
x_rsi = rsi_sample.shift(1).to_numpy()
reg = ols_stats(y_bps, x_rsi)
pd.DataFrame([{"alpha_bps": reg["alpha"], "beta": reg["beta"], "t_beta": reg["t_beta"],
               "p_beta": reg["p_beta"], "n": reg["n"], "r2": reg["r2"],
               "note": "RSI(5) Wilder on 510300 close; x=RSI(t-1); y=variant C g30 daily ret (bps)"}]
             ).to_csv(OUT / "15_rsi_regression.csv", index=False)
assert reg["n"] == int((~np.isnan(x_rsi)).sum())
print(f"BLOCK-15 ok | beta={reg['beta']:.3f} t={reg['t_beta']:.2f} p={reg['p_beta']:.4f} n={reg['n']}")


BLOCK-15 ok | beta=0.088 t=0.76 p=0.4501 n=2185


## BLOCK-16 Figure 10 成本敏感性

佣金率谱 {0, 0.005%, 0.01%, 0.02%, 0.03%} × 滑点 {0, 1, 2} tick 重跑变体 C（g30）。
输出：`16_cost_sensitivity.csv`、`16_fig10_cost.png`。

In [17]:
# BLOCK-16 成本敏感性
comm_list = [0.0, 0.00005, 0.0001, 0.0002, 0.0003]
slip_list = [0.0, 0.001, 0.002]
rows = []
for cm in comm_list:
    for sl in slip_list:
        res = run_backtest("g30", "band_vwap", dyn_lev=True, commission=cm, slippage=sl,
                           tag=f"cost_c{cm}_s{sl}")
        st = perf_stats(res["daily"]["ret"], res["daily"]["aum_end"])
        rows.append({"commission": cm, "slippage": sl, "TotalReturn": st["TotalReturn"],
                     "IRR": st["IRR"], "Sharpe": st["Sharpe"], "MDD": st["MDD"]})
COST = pd.DataFrame(rows)
zero = COST[(COST.commission == 0.0) & (COST.slippage == 0.0)].iloc[0]
print(f"zero-cost anchor: TR={zero['TotalReturn']:.2%} Sharpe={zero['Sharpe']:.3f}")
# 成本单调性：固定滑点，佣金升 → TR 不增
for sl in slip_list:
    sub = COST[COST.slippage == sl].sort_values("commission")["TotalReturn"].to_numpy()
    assert all(sub[i + 1] <= sub[i] + 1e-9 for i in range(len(sub) - 1)), sl
COST.to_csv(OUT / "16_cost_sensitivity.csv", index=False)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for sl, mk in zip(slip_list, ["o", "s", "^"]):
    sub = COST[COST.slippage == sl].sort_values("commission")
    ax.plot(sub["commission"] * 1e4, sub["TotalReturn"], marker=mk, label=f"slippage {sl * 1000:.0f} tick(s) x0.001")
ax.axvline(1.0, color="gray", ls=":", lw=1)
ax.set_xlabel("Commission per side (bps)")
ax.set_ylabel("Total Return")
ax.set_title("Fig10 Cost sensitivity (variant C, g30)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "16_fig10_cost.png")
plt.close(fig)
print("BLOCK-16 ok | monotonicity verified")
print(COST.to_string(index=False))


zero-cost anchor: TR=535.20% Sharpe=1.356
BLOCK-16 ok | monotonicity verified
 commission  slippage  TotalReturn     IRR  Sharpe     MDD
     0.0000    0.0000       5.3520  0.2375  1.3560 -0.1761
     0.0000    0.0010      -0.0776 -0.0093  0.0281 -0.3213
     0.0000    0.0020      -0.8665 -0.2072 -1.2700 -0.8665
     0.0001    0.0000       3.3268  0.1840  1.0907 -0.1994
     0.0001    0.0010      -0.3721 -0.0522 -0.2337 -0.4446
     0.0001    0.0020      -0.9092 -0.2416 -1.5227 -0.9092
     0.0001    0.0000       1.9468  0.1327  0.8255 -0.2232
     0.0001    0.0010      -0.5727 -0.0934 -0.4941 -0.5727
     0.0001    0.0020      -0.9382 -0.2745 -1.7728 -0.9382
     0.0002    0.0000       0.3663  0.0366  0.2969 -0.2690
     0.0002    0.0010      -0.8021 -0.1704 -1.0101 -0.8021
     0.0002    0.0020      -0.9714 -0.3361 -2.2663 -0.9714
     0.0003    0.0000      -0.3668 -0.0513 -0.2281 -0.4613
     0.0003    0.0010      -0.9084 -0.2408 -1.5183 -0.9084
     0.0003    0.0020      -0.9867 -0

## BLOCK-17 FAQ 扩展

Q4/Q24 月度收益表；Q5 多空腿权益；Q6 lookback 扫描；Q7 最差 10 季度；Q18 日内季节性；
Q19 空头 PnL ~ σ_SPY 回归；Q22 VWAP-only 止损；Q23 逐年每笔利润；Q12 扩展 ETF 简版。
输出：`17_*` 系列文件。

In [18]:
# BLOCK-17.1 Q4/Q24 月度收益表（变体 C g30 + B&H）
dlC2 = dlC.copy()
dlC2["ym"] = dlC2["date"].dt.to_period("M")
m_strat = dlC2.groupby("ym")["ret"].apply(lambda x: (1 + x).prod() - 1)
m_bh = pd.Series(ret_v, index=days).groupby(days.to_period("M")).apply(lambda x: (1 + x).prod() - 1)
mt = pd.DataFrame({"strat": m_strat, "bh": m_bh})
mt["year"] = mt.index.year
mt["month"] = mt.index.month
piv = mt.pivot(index="year", columns="month", values="strat")
piv.columns = [f"M{m:02d}" for m in piv.columns]
piv["YTD"] = mt.groupby("year")["strat"].apply(lambda x: (1 + x).prod() - 1)
piv["BH_YTD"] = mt.groupby("year")["bh"].apply(lambda x: (1 + x).prod() - 1)
piv.to_csv(OUT / "17_monthly_table.csv")
assert len(mt) == len(days.to_period("M").unique())
print(f"Q4/Q24 monthly table: {len(mt)} months x years {piv.shape[0]}")

# BLOCK-17.2 Q5 多空腿权益
rtC = RT[("C", "g30")].copy()
rtC["date"] = pd.to_datetime(rtC["date"])
pnl_long = rtC[rtC.direction == "long"].groupby("date")["pnl"].sum()
pnl_short = rtC[rtC.direction == "short"].groupby("date")["pnl"].sum()
eq_long = INIT_CAPITAL + pnl_long.reindex(days).fillna(0.0).cumsum()
eq_short = INIT_CAPITAL + pnl_short.reindex(days).fillna(0.0).cumsum()
pd.DataFrame({"date": days, "eq_long": eq_long.to_numpy(), "eq_short": eq_short.to_numpy(),
              "eq_both": RUNS[("C", "g30")]["equity"]["aum_end"].to_numpy()}
             ).to_csv(OUT / "17_long_short_legs.csv", index=False)
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(days, eq_long, label="Long leg", lw=1.1)
ax.plot(days, eq_short, label="Short leg", lw=1.1)
ax.plot(days, RUNS[("C", "g30")]["equity"]["aum_end"], label="Both", lw=1.1, color="black")
ax.set_yscale("log")
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Q5 Long/Short leg equity (variant C, g30)")
fig.tight_layout()
fig.savefig(OUT / "17_long_short_legs.png")
plt.close(fig)
print(f"Q5 legs: long end={eq_long.iloc[-1]:,.0f} short end={eq_short.iloc[-1]:,.0f}")

# BLOCK-17.3 Q6 lookback 扫描（噪声带 lookback；仓位 sigma14 固定 14）
lb_list = [5, 10, 14, 20, 25, 30, 60, 90]
rows = []
for lbk in lb_list:
    res = run_backtest("g30", "band_vwap", dyn_lev=True, lookback=lbk, tag=f"LB{lbk}")
    st = perf_stats(res["daily"]["ret"], res["daily"]["aum_end"])
    rows.append({"lookback": lbk, "TotalReturn": st["TotalReturn"], "IRR": st["IRR"],
                 "Vol": st["Vol"], "Sharpe": st["Sharpe"], "MDD": st["MDD"],
                 "N_Exec": len(res["trades"])})
LB_SW = pd.DataFrame(rows)
a14 = LB_SW[LB_SW.lookback == 14].iloc[0]
assert abs(a14["TotalReturn"] - base["TotalReturn"]) < 1e-9
LB_SW.to_csv(OUT / "17_lookback_sweep.csv", index=False)
print("Q6 lookback sweep:")
print(LB_SW.to_string(index=False))

# BLOCK-17.4 Q7 最差 10 季度（510300 B&H 季度收益）
q_bh = pd.Series(ret_v, index=days).groupby(days.to_period("Q")).apply(lambda x: (1 + x).prod() - 1)
q_strat = dlC2.groupby(dlC2["date"].dt.to_period("Q"))["ret"].apply(lambda x: (1 + x).prod() - 1)
qt = pd.DataFrame({"bh": q_bh, "strat": q_strat}).sort_values("bh")
worst10 = qt.head(10).reset_index()
worst10.columns = ["quarter", "bh_ret", "strat_ret"]
worst10.to_csv(OUT / "17_worst_quarters.csv", index=False)
print("Q7 worst 10 quarters (BH) vs strategy:")
print(worst10.to_string(index=False))

# BLOCK-17.5 Q18 日内季节性（按入场决策时点聚合平均每笔 PnL）
ent = rtC.copy()
sea = ent.groupby("entry_time").agg(n=("pnl", "size"), avg_pnl_bps=("ret_on_notional", lambda x: x.mean() * 1e4),
                                    avg_pnl_per_share=("pnl_per_share", "mean")).reset_index()
sea["order"] = sea["entry_time"].map(label_key)
sea = sea.sort_values("order").drop(columns="order")
sea.to_csv(OUT / "17_intraday_seasonality.csv", index=False)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.bar(sea["entry_time"], sea["avg_pnl_bps"])
ax.set_title("Q18 Intraday seasonality: avg round-trip PnL (bps) by entry decision time")
ax.set_ylabel("bps on notional")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "17_intraday_seasonality.png")
plt.close(fig)
print("Q18 seasonality:")
print(sea.to_string(index=False))

# BLOCK-17.6 Q19 空头整趟 PnL(bps) ~ 当日 sigma_SPY 回归
sh = rtC[rtC.direction == "short"].copy()
sh["sig"] = pd.Series(sigma14_v, index=days).reindex(sh["date"]).to_numpy()
reg19 = ols_stats(sh["ret_on_notional"].to_numpy() * 1e4, sh["sig"].to_numpy())
pd.DataFrame([{"alpha_bps": reg19["alpha"], "beta": reg19["beta"], "t_beta": reg19["t_beta"],
               "p_beta": reg19["p_beta"], "n": reg19["n"], "r2": reg19["r2"],
               "note": "short round-trip PnL(bps) ~ sigma_SPY,t (VIX proxy)"}]
             ).to_csv(OUT / "17_short_pnl_vol_regression.csv", index=False)
print(f"Q19 short PnL ~ sigma_SPY: beta={reg19['beta']:.1f} p={reg19['p_beta']:.3f} n={reg19['n']}")

# BLOCK-17.7 Q22 VWAP-only 止损变体
res_vw = run_backtest("g30", "vwap_only", dyn_lev=True, tag="Q22_vwap_only")
st_vw = perf_stats(res_vw["daily"]["ret"], res_vw["daily"]["aum_end"])
st_vw["N_Exec"] = len(res_vw["trades"])
pd.DataFrame([st_vw]).to_csv(OUT / "17_vwap_only_stats.csv", index=False)
print(f"Q22 VWAP-only: TR={st_vw['TotalReturn']:.2%} Sharpe={st_vw['Sharpe']:.3f} exec={st_vw['N_Exec']} "
      f"(vs band_vwap exec={len(RUNS[('C', 'g30')]['trades'])})")

# BLOCK-17.8 Q23 逐年平均每笔 PnL/Share
rtC["year"] = rtC["date"].dt.year
py = rtC.groupby("year").agg(avg_pnl_per_share=("pnl_per_share", "mean"),
                             n_trades=("pnl", "size")).reset_index()
py.to_csv(OUT / "17_profit_per_share_by_year.csv", index=False)
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.bar(py["year"].astype(str), py["avg_pnl_per_share"])
ax.set_title("Q23 Avg profit per share by year (variant C, g30)")
ax.set_ylabel("yuan / share")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "17_profit_per_share_by_year.png")
plt.close(fig)
print("Q23 by year:")
print(py.to_string(index=False))


Q4/Q24 monthly table: 108 months x years 9
Q5 legs: long end=976,210 short end=451,138


Q6 lookback sweep:
 lookback  TotalReturn     IRR    Vol  Sharpe     MDD  N_Exec
        5      -0.5247 -0.0822 0.1706 -0.4181 -0.5874    3608
       10      -0.5998 -0.1002 0.1695 -0.5387 -0.6230    3492
       14      -0.5727 -0.0934 0.1696 -0.4941 -0.5727    3410
       20      -0.6281 -0.1078 0.1679 -0.5960 -0.6312    3438
       25      -0.6229 -0.1063 0.1691 -0.5809 -0.6563    3376
       30      -0.5155 -0.0801 0.1671 -0.4170 -0.6002    3296
       60      -0.4946 -0.0756 0.1634 -0.4005 -0.5307    3214
       90      -0.3728 -0.0524 0.1662 -0.2413 -0.5073    3182
Q7 worst 10 quarters (BH) vs strategy:
quarter  bh_ret  strat_ret
 2022Q1 -0.1446    -0.0533
 2022Q3 -0.1420    -0.1070
 2018Q4 -0.1280     0.0334
 2020Q1 -0.1042    -0.0118
 2018Q2 -0.0921     0.0606
 2023Q4 -0.0704     0.0109
 2021Q3 -0.0611     0.0081
 2024Q4 -0.0498     0.0259
 2023Q2 -0.0415     0.0289
 2018Q1 -0.0340     0.0048
Q18 seasonality:
entry_time   n  avg_pnl_bps  avg_pnl_per_share
     10:00 678       3.

In [19]:
# BLOCK-17.9 Q12 扩展 ETF（510500/159915/588000，变体 C g30 简版）
# 复用主引擎：构建与各标的同构的数据上下文，临时替换模块级变量后运行，跑完恢复。
# 注：扩展标的未构建分钟级 tradability（BUYABLE/SELLABLE 全 True），仅作汇总指标参考。
CTX_KEYS = ["days", "n_days", "C_l", "O_l", "VW_l", "BUY_l", "SELL_l",
            "s14_l", "do_l", "pc_l", "SIGMA", "day_pos"]


def build_ext_ctx(tag):
    mn2 = pd.read_parquet(EXT_DIR / f"minute_bars_{tag}.parquet")
    mn2 = mn2[(mn2.trade_date >= SAMPLE_START) & (mn2.trade_date <= SAMPLE_END)].sort_values("trade_time")
    assert set(mn2.hhmm.unique()) <= set(LABELS)
    days2 = pd.DatetimeIndex(sorted(mn2.trade_date.unique()))

    def to_mat2(col):
        piv = mn2.pivot(index="trade_date", columns="hhmm", values=col)
        return piv.reindex(index=days2, columns=LABELS).to_numpy(dtype=float)

    O2, C2, V2, A2 = to_mat2("open"), to_mat2("close"), to_mat2("vol"), to_mat2("amount")
    fv = np.argmax(~np.isnan(O2), axis=1)
    do2 = np.where(np.isnan(O2[:, IDX["09:30"]]), O2[np.arange(len(days2)), fv], O2[:, IDX["09:30"]])
    dl2 = pd.read_csv(EXT_DIR / f"daily_{tag}.csv", parse_dates=["trade_date"]).sort_values("trade_date")
    dl2["ret"] = dl2["pct_chg"] / 100.0
    dl2["sigma14"] = dl2["ret"].rolling(LOOKBACK).std(ddof=1).shift(1)
    s2 = dl2.set_index("trade_date")
    MOVE2 = np.abs(C2[:, 1:] / do2[:, None] - 1.0)
    SIG2 = {LOOKBACK: pd.DataFrame(MOVE2).rolling(LOOKBACK).mean().shift(1).to_numpy()}
    CSA = np.nancumsum(np.where(np.isnan(A2), 0.0, A2), axis=1)
    CSV_ = np.nancumsum(np.where(np.isnan(V2), 0.0, V2), axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        VW2 = np.where(CSV_ > 0, CSA / (CSV_ * 100.0), np.nan)
    n2 = len(days2)
    buy2 = np.ones((n2, n_lab), dtype=bool).tolist()
    return {"days": days2, "n_days": n2, "C_l": C2.tolist(), "O_l": O2.tolist(),
            "VW_l": VW2.tolist(), "BUY_l": buy2, "SELL_l": [row[:] for row in buy2],
            "s14_l": s2["sigma14"].reindex(days2).to_numpy().tolist(),
            "do_l": do2.tolist(), "pc_l": s2["pre_close"].reindex(days2).to_numpy().tolist(),
            "SIGMA": SIG2, "day_pos": {d: i for i, d in enumerate(days2)}}


rows = []
for tag, nm in [("510500_SH", "510500.SH 中证500ETF"), ("159915_SZ", "159915.SZ 创业板ETF"),
                ("588000_SH", "588000.SH 科创50ETF")]:
    ctx = build_ext_ctx(tag)
    saved = {k: globals()[k] for k in CTX_KEYS}
    globals().update(ctx)
    try:
        res = run_backtest("g30", "band_vwap", dyn_lev=True, tag=f"Q12_{tag}")
    finally:
        globals().update(saved)
    st = perf_stats(res["daily"]["ret"], res["daily"]["aum_end"])
    st.update({"ETF": nm, "first_day": str(ctx["days"][0].date()), "days": int(ctx["n_days"]),
               "N_Exec": len(res["trades"])})
    rows.append(st)
    print(f"Q12 {nm}: days={ctx['n_days']} TR={st['TotalReturn']:.2%} Sharpe={st['Sharpe']:.3f}")
Q12 = pd.DataFrame(rows)[["ETF", "first_day", "days", "N", "TotalReturn", "IRR", "Vol",
                          "Sharpe", "HitRatio", "MDD", "N_Exec"]]
Q12.to_csv(OUT / "17_other_etfs.csv", index=False)
# 恢复后主上下文自检：主标的权益未变
assert abs(RUNS[("C", "g30")]["equity"]["nav"].iloc[-1] - (dlC["aum_end"].iloc[-1] / INIT_CAPITAL)) < 1e-12
print("Q12 done, main context restored")


Q12 510500.SH 中证500ETF: days=2186 TR=60.91% Sharpe=0.396


Q12 159915.SZ 创业板ETF: days=2185 TR=152.49% Sharpe=0.732
Q12 588000.SH 科创50ETF: days=1246 TR=-41.48% Sharpe=-0.562
Q12 done, main context restored


## BLOCK-18 标准交易流水与 A 股约束量化

- `trade_record.csv`：变体 C（g30）全部成交，前八列固定表头 + 审计列（UTF-8）
- 逐笔核验 tradability（可买/可卖、涨跌停范围、封板）；运行官方 validate_trade_record.py（≥3 探针：普通时点、15:00、涨停封板分钟）
- 底仓 T+0 可执行性量化：杠杆/名义本金分布 → 所需底仓+现金缓冲倍数；无底仓 T+1 对照
输出：`trade_record.csv`、`18_ashare_constraints.json`、`18_trade_validation.json`。

In [20]:
# BLOCK-18 交易流水 + 官方校验 + A股约束量化
trC = RUNS[("C", "g30")]["trades"].copy()
rec = pd.DataFrame({
    "交易日期": pd.to_datetime(trC["date"]).dt.strftime("%Y%m%d"),
    "交易时间": trC["exec_time"].str.replace(":", ""),
    "股票代码": TS_CODE,
    "股票名称": TS_NAME,
    "交易价格": trC["price"].round(3),
    "交易数量": trC["shares"].astype(int),
    "交易方向": trC["side"],
    "交易标识": [f"{RUN_ID}-C30-{i:06d}" for i in range(1, len(trC) + 1)],
    "信号时间": trC["decision_time"],
    "止损类型": trC["reason"].replace("", "ENTRY"),
    "佣金": trC["commission"].round(4),
    "滑点": trC["slippage_cost"].round(4),
    "费用合计": trC["cost"].round(4),
    "杠杆": trC["lev"].round(4),
    "策略变体": "C(band_vwap+dynamic,g30)",
    "run_id": RUN_ID,
})
assert rec["交易标识"].is_unique
assert (rec["交易数量"] % 100 == 0).all() and (rec["交易数量"] > 0).all()
rec.to_csv(ROOT / "trade_record.csv", index=False, encoding="utf-8")

# 逐笔可交易性核验（含 15:00 强平）
chk = rec.merge(trad[["交易日期", "交易时间", "可买", "可卖", "涨停价", "跌停价", "涨停封板", "跌停封板", "是否有成交"]],
                on=["交易日期", "交易时间"], how="left")
assert chk["可买"].notna().all(), "存在 tradability 缺失的成交时点"
is_buy = chk["交易方向"] == "买入"
viol_buy = int(((chk["可买"] != 1) & is_buy).sum())
viol_sell = int(((chk["可卖"] != 1) & ~is_buy).sum())
viol_px = int(((chk["交易价格"] > chk["涨停价"] + 1e-9) | (chk["交易价格"] < chk["跌停价"] - 1e-9)).sum())
viol_seal = int(((chk["涨停封板"] == 1) & is_buy).sum() + ((chk["跌停封板"] == 1) & ~is_buy).sum())
viol_act = int((chk["是否有成交"] != 1).sum())
assert viol_buy == 0 and viol_sell == 0 and viol_px == 0 and viol_seal == 0 and viol_act == 0,     (viol_buy, viol_sell, viol_px, viol_seal, viol_act)
print(f"tradability 核验: {len(chk)} 笔全部合规 (buy/sell/price/seal violations = 0)")

# 官方校验器（>=3 探针：普通时点、15:00 强平时点、涨停封板分钟）
seal_row = trad[trad["涨停封板"] == 1].iloc[0]
probes = [f"{TS_CODE},{rec['交易日期'].iloc[0]},{rec['交易时间'].iloc[0]}",
          f"{TS_CODE},{rec[rec['交易时间'] == '1500']['交易日期'].iloc[0]},1500",
          f"{TS_CODE},{seal_row['交易日期']},{seal_row['交易时间']}"]
validator = ROOT.parent / ".github" / "harness" / "validators" / "validate_trade_record.py"
cmd = [sys.executable, str(validator), "--trades", str(ROOT / "trade_record.csv"),
       "--tradability", str(PROC / "tradability.csv")]
for p_ in probes:
    cmd += ["--probe", p_]
cp = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
val_out = {"returncode": cp.returncode, "probes": probes,
           "stdout_tail": cp.stdout[-2000:], "stderr_tail": cp.stderr[-1000:]}
dump_json(val_out, "18_trade_validation.json")
print("validate_trade_record returncode:", cp.returncode)
print(cp.stdout[-600:])
assert cp.returncode == 0, "official trade-record validator FAILED"

# A 股约束量化（底仓 T+0 可执行性）
lev_s = dlC["lev"].dropna()
rtC2 = RT[("C", "g30")]
constraints = {
    "mode": "EXECUTABLE_WITH_INVENTORY_T0",
    "assumption": ("用户 2026-10-04 裁决：账户已持有足额底仓与现金缓冲；多头腿先买(现金)后卖(旧份额)、"
                   "空头腿先卖(底仓)后买(资金回笼)，日内均可合法闭合；不使用融券"),
    "buffer_requirement": {
        "long_leg": "现金缓冲 >= 当日名义本金 = lev x AUM",
        "short_leg": "底仓缓冲(份额) >= 当日名义本金 = lev x AUM",
        "note": "缓冲倍数 = 当日杠杆；下表给出杠杆分布"},
    "lev_distribution": {
        "mean": float(lev_s.mean()), "median": float(lev_s.median()),
        "p95": float(lev_s.quantile(0.95)), "max": float(lev_s.max()),
        "pct_days_lev_gt_1": float((lev_s > 1).mean()),
        "pct_days_lev_gt_2": float((lev_s > 2).mean()),
        "pct_days_lev_gt_3": float((lev_s > 3).mean()),
        "pct_days_lev_eq_4cap": float((lev_s >= MAX_LEV - 1e-12).mean())},
    "round_trips": int(len(rtC2)),
    "short_leg_share": float((rtC2["direction"] == "short").mean()),
    "executions": int(len(rec)),
    "tradability_violations": {"buy": viol_buy, "sell": viol_sell, "price": viol_px, "seal": viol_seal,
                               "zero_activity": viol_act},
    "seal_minutes_in_sample": {"up": int((trad["涨停封板"] == 1).sum()), "down": int((trad["跌停封板"] == 1).sum())},
    "t1_counterfactual": ("无底仓纯 T+1 制度下：当日买入份额不可当日卖出，全部 "
                          f"{len(rtC2)} 趟日内往返均不可闭合（占比 100%）；空头腿无融券亦不可行。"
                          "故本策略在 A 股的可执行性完全依赖底仓 T+0 假设，"
                          "且缓冲需覆盖最高 {:.2f}x 名义本金".format(float(lev_s.max()))),
}
dump_json(constraints, "18_ashare_constraints.json")
print("BLOCK-18 ok")
print(json.dumps(constraints["lev_distribution"], ensure_ascii=False, indent=2))


tradability 核验: 3410 笔全部合规 (buy/sell/price/seal violations = 0)


validate_trade_record returncode: 0
PROBE 510300.SH 20170123 1001 可买=1 可卖=1 停牌=0
PROBE 510300.SH 20170210 1500 可买=1 可卖=1 停牌=0
PROBE 510300.SH 20241008 0930 可买=0 可卖=1 停牌=0
PASS: 3410 条交易记录通过基础校验

BLOCK-18 ok
{
  "mean": 2.1969178503402436,
  "median": 2.1121568063379925,
  "p95": 4.0,
  "max": 4.0,
  "pct_days_lev_gt_1": 0.938700823421775,
  "pct_days_lev_gt_2": 0.5585544373284538,
  "pct_days_lev_gt_3": 0.15599268069533395,
  "pct_days_lev_eq_4cap": 0.05901189387008234
}


## 结果速览（g30 论文主线）

- 变体 A/B/C 与 B&H 汇总指标见 `output/08_summary_stats.csv`；alpha/beta 见 `08_alpha_beta.csv`
- 全部产物清单：output/01~18 系列 + trade_record.csv + output/ext（Q12 扩展数据）
- 与论文数值的逐项对比见 `reports/comparison-report.md`（阶段 6 产出）

In [21]:
# 结果速览
print("=== Summary (g30 + BH) ===")
print(SUMMARY[SUMMARY.Grid.isin(["g30", "-"])][["Variant", "Grid", "TotalReturn", "IRR", "Vol",
      "Sharpe", "HitRatio", "MDD"]].to_string(index=False))
print()
print("=== Alpha/Beta (g30) ===")
print(ALPHA_BETA[ALPHA_BETA.Grid == "g30"][["Variant", "alpha_annual", "beta", "t_alpha", "p_alpha"]]
      .to_string(index=False))
print()
print("MAIN NOTEBOOK RUN COMPLETE")


=== Summary (g30 + BH) ===
Variant Grid  TotalReturn     IRR    Vol  Sharpe  HitRatio     MDD
      A  g30       0.2228  0.0235 0.1078  0.2689    0.3038 -0.2329
      B  g30      -0.1655 -0.0206 0.0881 -0.1931    0.2274 -0.2921
      C  g30      -0.5727 -0.0934 0.1696 -0.4941    0.2274 -0.5727
     BH    -       0.6427  0.0589 0.1892  0.3970    0.5041 -0.4216

=== Alpha/Beta (g30) ===
Variant  alpha_annual   beta  t_alpha  p_alpha
      A        0.0252 0.0505   0.6906   0.4898
      B       -0.0215 0.0600  -0.7252   0.4683
      C       -0.0910 0.0958  -1.5885   0.1122

MAIN NOTEBOOK RUN COMPLETE
